# Compare embedding spaces across models

**What you'll learn.** Different foundation models represent the same genes
very differently. embpy lets you embed one set of entities several ways and
then *measure* how much the models agree — with nearest-neighbour overlap,
cross-space correlation, and side-by-side projections.

The practical question this answers: **does my choice of model actually
matter for these entities?** (It usually does.)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, tl, pl

embedder = BioEmbedder(device="auto", organism="human")

genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1",
         "GATA3", "FOXP3", "CD8A", "IL2"]
adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)

## Embed the same genes several ways

We give each model its own `.obsm` key. Here: a prior-knowledge table
(`genept`), a DNA sequence model (`hyenadna_small_32k`), and a text model
(`minilm_l6_v2`) applied to the gene symbols. Each stores an embedding matrix
aligned to the same 12 genes.

In [ ]:
specs = [
    ("genept",             "X_prior", {}),
    ("hyenadna_small_32k", "X_dna",   {"region": "exons"}),
    ("minilm_l6_v2",       "X_text",  {}),
]
for model, key, extra in specs:
    adata = embedder.embed(
        adata, entity_type="gene", id_type="symbol", obs_column="symbol",
        model=model, output="anndata", key=key, **extra,
    )

print("embedding spaces:", [k for k in adata.obsm if k.startswith("X_")])

## How much do they agree? Nearest-neighbour overlap

The most interpretable comparison: for each gene, how many of its *k* nearest
neighbours are shared between two embedding spaces? An overlap near 1 means the
models organise these genes the same way; near 0 means they disagree entirely.

In [ ]:
_, mean_overlap = tl.compute_knn_overlap(adata, "X_prior", "X_dna", k=4)
print(f"prior-knowledge vs DNA — mean 4-NN overlap: {mean_overlap:.3f}")

pl.knn_overlap(adata, obsm_keys=["X_prior", "X_dna", "X_text"], k=4)

## Are the geometries correlated?

`cross_embedding_correlation` compares the *pairwise-distance* structure of two
spaces: if genes that are far apart in one model are also far apart in the
other, the correlation is high even when the raw vectors are unrelated.

In [ ]:
pl.cross_embedding_correlation(adata, "X_prior", "X_text", method="pearson")

## Mind the scale

Embeddings from different models live at wildly different magnitudes. If you
ever mix them, this is why you standardise first — see it directly:

In [ ]:
pl.embedding_norms(adata, obsm_keys=["X_prior", "X_dna", "X_text"])

## See it — side-by-side projections

Finally, project each space to 2-D. The same 12 genes, three models: clusters
that hold together in one view can scatter in another.

In [ ]:
for key in ["X_prior", "X_dna", "X_text"]:
    pl.plot_embedding_space(
        adata, obsm_key=key, method="pca", annotate=True, annotate_col="symbol",
        title=f"gene embeddings — {key}",
    )

## Takeaway

Models genuinely disagree, so *which* embedding you pick changes your
downstream result. The next notebook turns this from a qualitative comparison
into a number: **which model is actually best for a task you care about.**

**Next:** [Which model captures my biology?](04_benchmark_models.ipynb)